# Narrative RAG: real data, training, retrieval and citation audit

This end-to-end notebook orchestrates the authoritative `src` modules. It includes
real data preparation, validation retrieval, optional actual model training,
saved-model answers, source citation checks and plots. Model training is now an
explicit opt-in notebook path at the user's request; core implementation remains
in Python modules and root `train.py`.

Opening or running the default cells does **not** download weights, train a model,
open test QA, overwrite artifacts or rescore a test. Outputs are intentionally
empty in this distributed notebook. Saved historical aggregates are clearly
labelled; newly enabled inference cells create real outputs in ignored folders.

Use a fresh virtual environment/kernel for this repository. Installing the other
research repositories into the same environment is unsupported (`src` name and
optional dependency collisions). Install `pip install -e ".[reader]"` and, for
plots/notebook use, `pip install jupyter matplotlib`. Cloud runtime versions and
model pins are recorded in the study protocol and actual smoke receipt.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
import hashlib
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/narrative_training.py").is_file(), "Run from this repository or its notebooks folder"
sys.path.insert(0, str(ROOT))
PROTOCOL = ROOT / "docs/v5-training-protocol.json"
RIGHTS = ROOT / "docs/v5-source-rights.json"
STAGED = ROOT / "data/interim/v5-fairytale-cpu"
RUN_DIR = ROOT / "models" / ("notebook-run-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN_TRAIN = False
RUN_RETRIEVAL = False
RUN_PLOTS = False
RUN_ANSWER = False
RUN_TEST = False
print("Repository located; optional training/inference/test paths are disabled.")

In [ ]:
protocol = json.loads(PROTOCOL.read_text(encoding="utf-8"))
rights = json.loads(RIGHTS.read_text(encoding="utf-8"))
print("Dataset revision:", protocol["dataset"]["revision"])
print("Generator:", protocol["generator"]["name"], protocol["generator"]["revision"])
print("Validation origins:", protocol["dataset"]["validation_origins"])
print("Test origins (QA not parsed here):", protocol["dataset"]["test_origins"])
print("Dataset license:", rights["dataset_license"])

## Acquire and stage licensed real data

FairytaleQA provides human-authored questions and evidence-section links. We use
its pinned Apache-2.0 source privately. The dataset's origin labels are not always
exact source editions: all Lang-origin groups stay together in validation.
Only official training stories from eligible other families enter training.
Potter and Grimm are held out. Recurring plots and pretraining exposure remain
possible; this is not a standard-benchmark reproduction.

The source archive contains test CSV bytes, but neither this preparation module
nor the training runner parses test QA. Preparation stages it separately. The
training Kaggle dataset excludes test QA entirely. Enable acquisition/preparation
only when needed; use a new output folder rather than replacing a study.

In [ ]:
FETCH_SOURCE = False
PREPARE_SOURCE = False
revision = protocol["dataset"]["revision"]
external = ROOT / "data/external/fairytaleqa"
archive_path = external / "source.zip"
source_path = external / ("FairytaleQAData-" + revision)
if FETCH_SOURCE:
    import urllib.request
    import zipfile
    external.mkdir(parents=True, exist_ok=True)
    if not archive_path.exists():
        url = "https://codeload.github.com/uci-soe/FairytaleQAData/zip/" + revision
        archive_path.write_bytes(urllib.request.urlopen(url, timeout=120).read())
    expected = "353ec962a911ea9b2a4fbba2fe9fdebe2a92c392ce4be36194f2062f0ac9a998"
    assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == expected, "Pinned archive bytes changed"
    if not source_path.exists():
        with zipfile.ZipFile(archive_path) as handle:
            handle.extractall(external)
if PREPARE_SOURCE:
    from src.data.prepare_fairytale import prepare
    # Existing output is deliberately rejected by the module.
    prepare(source_path, STAGED, RIGHTS, PROTOCOL)
print("Source acquisition/preparation enabled:", FETCH_SOURCE, PREPARE_SOURCE)

In [ ]:
from collections import Counter
manifest_path = STAGED / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("Source-only story counts:", dict(Counter(r["split"] for r in manifest["stories"])))
    print("Removed train source duplicates:", len(manifest["removed_train_duplicates"]))
    print("This cell reads source manifest metadata, not test questions or answers.")
else:
    print("No private staged data present; enable the explicit preparation path when needed.")

## Validation retrieval and historical diagnostics

BM25 retrieves four sections from the question's given story. Story routing is
supplied by the dataset, so this is not global book identification or cross-book
thematic evaluation. Gold section IDs are used only for scoring/training, never
as retrieval inputs. Evidence recall/precision are **not answer EM or F1**.
Validation sampling is deterministic by canonical CSV row identity, with 100
questions per validation origin. Unused validation QA never enter training.

In [ ]:
if RUN_RETRIEVAL:
    from src.narrative_retrieval_baseline import run as evaluate_retrieval
    destination = ROOT / "data/interim" / (RUN_DIR.name + "-retrieval.json")
    evaluate_retrieval(STAGED, destination)
else:
    print("Retrieval scoring disabled; reading a recorded aggregate only.")
recorded_retrieval = json.loads((ROOT / "docs/v5-validation-retrieval-baseline.json").read_text(encoding="utf-8"))
print("RECORDED VALIDATION EVIDENCE METRICS, no new QA inference:",
      {k: recorded_retrieval[k] for k in ("n", "recall_at_4", "precision_at_4", "test_accessed")})

In [ ]:
historical = json.loads((ROOT / "docs/v4-cached-validation-results.json").read_text(encoding="utf-8"))
print("HISTORICAL cached Dorian study; not the new source-family cohort:")
print(historical["scope"])
print("Promotion winner:", historical["qualified_winner"])
print("Historical policies retained:", len(historical["results"]))

## Optional actual fine-tuning

The command below performs real supervised T5-small training, using training
question/answer and annotated evidence labels. Primary validation retrieval does
not inject gold evidence. It requires the real private data and pinned pretrained
weights; it has no mock model or synthetic fallback. On a local machine, enable
only when sufficient CPU/RAM and time are available. The recorded study uses a
free Kaggle CPU environment after account GPU quota exhaustion. Its plain runner
imports versioned Python files from a separate private dataset.

The original BART/1024-token plan was prospectively adapted to T5-small/512 tokens
before model scoring. This is a feasibility change, not a BART-versus-T5 result.
The default three epochs select by validation token F1, with earlier-epoch ties.
Do not tune from the held-out test. A new timestamped output path prevents silent
replacement of training evidence.

In [ ]:
if RUN_TRAIN:
    from src.narrative_training import run as train_narrative
    train_narrative(SimpleNamespace(data=STAGED, output=RUN_DIR, protocol=PROTOCOL,
                                   micro_batch=2, threads=4, smoke=False))
else:
    print("Actual training disabled. Set RUN_TRAIN explicitly, or inspect a real downloaded run.")
print("Fresh output path:", RUN_DIR.name)

In [ ]:
# To inspect an existing real run, explicitly set RUN_DIR to its ignored local folder.
validation_path = RUN_DIR / "validation-results.json"
if validation_path.exists():
    learning_curve = json.loads(validation_path.read_text(encoding="utf-8"))
    print("REAL SAVED TRAINING METRICS:")
    for row in learning_curve:
        print({"epoch": row["epoch"], "answer_f1": row["validation"]["f1"],
               "answer_em": row["validation"]["em"], "n": row["validation"]["n"]})
else:
    learning_curve = []
    print("No newly selected training artifact at this path; no training score invented.")

## Plots from actual saved aggregates

These optional plots read recorded evidence diagnostics and, if present, the real
run's validation learning curve. They do not generate example performance values.
Plots are saved outside Git. Evidence section presence after context truncation
is not proof that every relevant sentence or generated claim is retained.

In [ ]:
if RUN_PLOTS:
    import matplotlib.pyplot as plt
    figure_dir = ROOT / "data/interim/notebook-figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    origins = recorded_retrieval["per_origin"]
    fig, axis = plt.subplots(figsize=(8, 4))
    axis.bar(list(origins), [v["recall_at_4"] for v in origins.values()])
    axis.set_ylabel("Annotated section recall@4")
    axis.set_title("Recorded validation evidence retrieval (not QA F1)")
    axis.set_ylim(0, 1)
    axis.tick_params(axis="x", labelrotation=30)
    fig.tight_layout()
    fig.savefig(figure_dir / "validation-evidence.png", dpi=160)
    plt.show()
    if learning_curve:
        fig, axis = plt.subplots()
        axis.plot([r["epoch"] for r in learning_curve],
                  [r["validation"]["f1"] for r in learning_curve], marker="o")
        axis.set(xlabel="Epoch (0 = untuned)", ylabel="Actual validation answer token F1")
        fig.tight_layout()
        fig.savefig(figure_dir / "training-validation.png", dpi=160)
        plt.show()
else:
    print("Plot execution disabled; all plotted values would come from saved real aggregates.")

## Saved-model answers and exact citation audit

Set a real selected `RUN_DIR`, a processed story file and your question. The CLI
checks frozen selection, provenance and trained weight hashes. Its experimental
schema labels an answer as **generated**, separately from retrieved source spans.
Exact source citations establish location; they do not establish entailment.
This answerable-only study does not calibrate unknown-question abstention.
The existing extractive reader and its audited span schema remain separate.

In [ ]:
STORY_PATH = None  # Set to a real *.story.json under the ignored staged source.
QUESTION = ""       # Set an explicit question; no fabricated demonstration answer.
ANSWER_PATH = ROOT / "data/interim" / (RUN_DIR.name + "-answer.json")
if RUN_ANSWER:
    assert STORY_PATH is not None and QUESTION.strip(), "Choose real source and question"
    subprocess.run([sys.executable, "-m", "evaluation_scripts.predict_narrative",
                    "--run", str(RUN_DIR), "--story", str(STORY_PATH),
                    "--question", QUESTION, "--output", str(ANSWER_PATH)],
                   cwd=ROOT, check=True)
else:
    print("Saved-model inference disabled; no weights loaded by this cell.")

In [ ]:
if RUN_ANSWER and ANSWER_PATH.exists():
    from src.models.narrative_generator import verify_story, verify_run
    story = json.loads(Path(STORY_PATH).read_text(encoding="utf-8"))
    verify_story(story)
    verify_run(RUN_DIR)
    saved = json.loads(ANSWER_PATH.read_text(encoding="utf-8"))
    assert saved["answer_kind"] == "generated"
    assert saved["source_sha256"] == story["source_sha256"]
    citations = saved["retrieved_evidence"]
    exact = all(story["text"][c["start"]:c["end"]] == c["text"] for c in citations)
    assert exact and citations, "Nonempty exact source evidence required"
    print("Actual saved artifact source citations valid:", exact, "count:", len(citations))
    print("Generated text is kept in the private artifact; this check is not an entailment score.")
else:
    print("No newly generated private artifact requested; source audit skipped explicitly.")

## Explicit one-shot held-out evaluation

This cell stays disabled. It is only appropriate after validation experiments
are complete and the model/policy is frozen. The evaluation module writes an
exclusive test-started marker before parsing test QA and refuses overwrite. A
copied artifact folder does not make a previously observed test fresh: any later
reproduction must be labelled retrospective. Dorian, Persuasion, Call of the Wild,
Journey and Cask are already observed historical books, never fresh test sources.

Test questions/answers must be made available locally only for this explicit step;
they are absent from the training cloud dataset. Never add them to a notebook's
committed outputs. The current training study has not reported a held-out score.

In [ ]:
if RUN_TEST:
    assert (RUN_DIR / "selection.json").exists(), "Freeze validation selection first"
    assert not (RUN_DIR / "test-started.json").exists(), "Existing test cannot be silently rescored"
    subprocess.run([sys.executable, "-m", "evaluation_scripts.evaluate_narrative",
                    "--data", str(STAGED), "--run", str(RUN_DIR)], cwd=ROOT, check=True)
else:
    print("Held-out QA remains closed. No test scoring performed by default.")

## Interpretation

Report answer EM/token F1 separately from evidence relevance, exact source citation
validity, and any future manual entailment audit. Provide origin/story counts,
reference-count strata, explicit versus implicit questions and narrative attributes.
Children's story adaptation is not evidence of adult novel or cross-book reasoning.
Fairy-tale plots may cross collections and appear in pretraining. Preserve failures,
negative ablations, source hashes and hardware changes rather than forcing a gain.

The original notebooks remain useful EDA. This notebook adds the explicitly
requested end-to-end orchestration while keeping normal Python source authoritative.